# Setup

In [ ]:
import sys
sys.path.append("/home/habjan.e/TNG/Codes/TNG_workshop")
sys.path.append("/home/habjan.e/TNG/TNG_cluster_dynamics")
sys.path.append("/home/habjan.e/TNG/Codes/Fourier_Analysis")

import warnings

import numpy as np
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution

import TNG_DA
import iapi_TNG as iapi

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "text.usetex": False,
})

# ------------------------------------------------------------------ #
# Paths
# ------------------------------------------------------------------ #
TNG_data_path    = '/home/habjan.e/TNG/Data/'
bahamas_path     = '/projects/mccleary_group/habjan.e/TNG/Data'
base_coh_3d_path = bahamas_path + '/coherence_data_3D'   # 3D power spectra + coherence
base_coh_path    = bahamas_path + '/coherence_data'      # 2D (per-projection) coherence
figure_path      = '/home/habjan.e/TNG/TNG_cluster_dynamics/power_ratio_figures'

h = 0.6774

cluster_ids     = [f"{i:03d}" for i in range(1, 101)]   # BAHAMAS
cluster_ids_tng = [f"{i:01d}" for i in range(0, 100)]   # TNG300-*

# ------------------------------------------------------------------ #
# Simulation registry. Insertion order sets the plotting and legend order.
#   xsec : sigma/m in cm^2 g^-1, or None if the model has no single value
#          (vdSIDM), in which case it is left out of the cross-section panel.
# ------------------------------------------------------------------ #
SIMS = {
    'CDMb':     dict(label=r'BAHAMAS CDM',
                     color='red',      xsec=0.0,  marker='D', ms=10, ids=cluster_ids),
    'SIDM0.1b': dict(label='BAHAMAS SIDM' + '\n' + r'0.1 cm$^2$ g$^{-1}$',
                     color='green',    xsec=0.1,  marker='D', ms=10, ids=cluster_ids),
    'SIDM0.3b': dict(label='BAHAMAS SIDM' + '\n' + r'0.3 cm$^2$ g$^{-1}$',
                     color='orange',   xsec=0.3,  marker='D', ms=10, ids=cluster_ids),
    'SIDM1b':   dict(label='BAHAMAS SIDM' + '\n' + r'1.0 cm$^2$ g$^{-1}$',
                     color='black',    xsec=1.0,  marker='D', ms=10, ids=cluster_ids),
    'vdSIDMb':  dict(label='BAHAMAS' + '\n' + r'Velocity-Dependent SIDM',
                     color='blue',     xsec=None, marker='D', ms=10, ids=cluster_ids),
    'TNG300-1': dict(label='TNG300-1 CDM',
                     color='purple',   xsec=0.0,  marker='o', ms=7,  ids=cluster_ids_tng),
    'TNG300-2': dict(label='TNG300-2 CDM',
                     color='deeppink', xsec=0.0,  marker='o', ms=7,  ids=cluster_ids_tng),
    'TNG300-3': dict(label='TNG300-3 CDM',
                     color='olive',    xsec=0.0,  marker='o', ms=7,  ids=cluster_ids_tng),
}

ALL_MODELS  = tuple(SIMS)
TNG_MODELS  = ('TNG300-1', 'TNG300-2', 'TNG300-3')
XSEC_MODELS = tuple(m for m in SIMS if SIMS[m]['xsec'] is not None)

# Figure 1: 3D power ratio versus cross-section

Left panel: the dark-matter/gas cross-power ratio $R^{\rm 3D}_{\rm mg}$ as a function of
scale. Right panel: $R^{\rm 3D}_{\rm mg}$ averaged over a window in $\theta/R_{200}$,
against $\sigma/m$.

The window $[\theta_{\rm lo}, \theta_{\rm hi}]$ and the coherence-length cut
$[\ell_{\rm lo}, \ell_{\rm hi}]$ used to select clusters are fitted in the
optimization cell below rather than hand-picked.

### Import 3D power spectra and coherence lengths

In [ ]:
def _load_power_stack(model, ids):
    """Per-cluster 3D power spectra and coherence for one simulation.

    Each entry has shape (Ncluster, Ntheta); `th` is theta / R_200.
    """
    def load(stem):
        return np.array([np.load(f'{base_coh_3d_path}/{model}/{stem}_{i}.npy') for i in ids])

    return dict(
        coh=load('coh'),
        pm=load('power_mass'),
        pg=load('power_gas'),
        pc=load('power_cross'),
        th=load('theta'),
    )


def _load_coh_length(model, ids):
    """Per-cluster 3D coherence length, shape (Ncluster,)."""
    return np.array([
        np.load(f'{base_coh_3d_path}/{model}/coherence_length_{i}.npy')[0] for i in ids
    ])


POWER = {}
for _model, _cfg in SIMS.items():
    POWER[_model] = _load_power_stack(_model, _cfg['ids'])
    POWER[_model]['coh_3d'] = _load_coh_length(_model, _cfg['ids'])

for _model in ALL_MODELS:
    print(f"{_model:10s} clusters = {POWER[_model]['pm'].shape[0]:3d}   "
          f"theta samples = {POWER[_model]['th'].shape[1]:3d}")

### Functions

In [ ]:
# Shared theta/R_200 grid for the spectrum panel.
theta_grid = np.logspace(-2.2, 1.2, 30)

# Hand-picked window and coherence cut used before the optimizer existed. Kept as
# the optimizer's initial guess and as a fallback for the plotting cell.
#   (theta_lo, theta_hi, low_coh, up_coh)
X0_PARAMS = (5 * 10**-2, 7 * 10**-2, 0.05, 0.4)


def select_clusters(coh_3d, low_coh, up_coh):
    """Indices of the clusters whose 3D coherence length lies in (low_coh, up_coh)."""
    return np.where((coh_3d > low_coh) & (coh_3d < up_coh))[0]


_CURVE_CACHE = {}


def _cluster_curves(model, kind='cross_over_gas'):
    """Per-cluster (theta, y) curves: sorted, de-duplicated, finite-only.

    kind : 'cross_over_gas', 'mass_over_gas', 'cross_over_mass' or 'coherence'.

    Cached, because every optimizer step re-interpolates the same curves.
    """
    key = (model, kind)
    if key in _CURVE_CACHE:
        return _CURVE_CACHE[key]

    d = POWER[model]

    with np.errstate(divide='ignore', invalid='ignore'):
        if kind == 'cross_over_gas':
            y_all = d['pc'] / d['pg']
        elif kind == 'mass_over_gas':
            y_all = d['pm'] / d['pg']
        elif kind == 'cross_over_mass':
            y_all = d['pc'] / d['pm']
        elif kind == 'coherence':
            y_all = d['coh']
        else:
            raise ValueError(f'unknown kind={kind!r}')

    th = d['th']
    curves = []

    for i in range(th.shape[0]):
        order = np.argsort(th[i])
        x, y = th[i][order], y_all[i][order]

        good = np.isfinite(x) & np.isfinite(y)
        x, y = x[good], y[good]

        x_u, idx_u = np.unique(x, return_index=True)
        curves.append((x_u, y[idx_u]) if x_u.size >= 2 else (np.empty(0), np.empty(0)))

    _CURVE_CACHE[key] = curves
    return curves


def interp_curves(model, theta_eval, idx=None, kind='cross_over_gas'):
    """Interpolate per-cluster curves onto `theta_eval`.

    Returns (Nselected, len(theta_eval)). Points outside a cluster's own theta
    range stay NaN, so no cluster is ever extrapolated.
    """
    curves = _cluster_curves(model, kind=kind)
    if idx is None:
        idx = np.arange(len(curves))

    theta_eval = np.atleast_1d(np.asarray(theta_eval, dtype=float))
    out = np.full((len(idx), theta_eval.size), np.nan)

    for row, i in enumerate(idx):
        x, y = curves[i]
        if x.size < 2:
            continue
        in_range = (theta_eval >= x[0]) & (theta_eval <= x[-1])
        out[row, in_range] = np.interp(theta_eval[in_range], x, y)

    return out


def _nanmean(a, axis=0):
    """np.nanmean without the all-NaN RuntimeWarning (all-NaN slices give NaN)."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return np.nanmedian(a, axis=axis)


def band_average(model, theta_lo, theta_hi, idx=None, kind='cross_over_gas', n_band=16):
    """Per-cluster mean of the curve over theta/R_200 in [theta_lo, theta_hi].

    The average is taken on a dense log-spaced sub-grid inside the window, not on
    the native theta sampling: each cluster carries only ~30 theta points on its
    own R_200-scaled grid, so a window this narrow contains ~1 native sample and
    the result -- hence the optimizer's loss -- would be a step function of
    theta_lo / theta_hi.
    """
    band = np.logspace(np.log10(theta_lo), np.log10(theta_hi), n_band)
    return _nanmean(interp_curves(model, band, idx=idx, kind=kind), axis=1)


def bootstrap_ci(vals, n_boot=1000, ci=(16, 84), random_state=123):
    """Mean over clusters (axis 0) with a bootstrap CI on that mean.

    Accepts 1D (one value per cluster) or 2D (clusters x theta) input.
    """
    vals = np.asarray(vals, dtype=float)
    rng = np.random.default_rng(random_state)
    n = vals.shape[0]

    center = _nanmean(vals, axis=0)
    boot = np.empty((n_boot,) + np.shape(center))

    for b in range(n_boot):
        boot[b] = _nanmean(vals[rng.integers(0, n, size=n)], axis=0)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        lo = np.nanpercentile(boot, ci[0], axis=0)
        hi = np.nanpercentile(boot, ci[1], axis=0)

    return center, lo, hi


def band_summary(params, models=ALL_MODELS, kind='cross_over_gas', n_band=16,
                 n_boot=1000, err_method='bootstrap', random_state=123):
    """<R^3D_mg> over the theta window, per simulation, for the selected clusters.

    params : (theta_lo, theta_hi, low_coh, up_coh)

    Returns {model: dict(center, lo, hi, sigma, n)}, where `sigma` is the
    symmetrized 1-sigma uncertainty on the mean -- the quantity the loss
    functions weight by -- and `n` is the number of selected clusters.

    err_method : 'bootstrap' (matches the plotted error bars) or 'sem'
        (analytic standard error; smoother, and cheaper, for the optimizer).
    """
    theta_lo, theta_hi, low_coh, up_coh = params
    out = {}

    for m in models:
        idx = select_clusters(POWER[m]['coh_3d'], low_coh, up_coh)
        vals = band_average(m, theta_lo, theta_hi, idx=idx, kind=kind, n_band=n_band)
        vals = vals[np.isfinite(vals)]

        if vals.size < 2:
            out[m] = dict(center=np.nan, lo=np.nan, hi=np.nan,
                          sigma=np.nan, n=int(vals.size))
            continue

        if err_method == 'sem':
            center = float(np.mean(vals))
            sigma = float(np.std(vals, ddof=1) / np.sqrt(vals.size))
            lo, hi = center - sigma, center + sigma
        elif err_method == 'bootstrap':
            center, lo, hi = bootstrap_ci(vals, n_boot=n_boot, random_state=random_state)
            center, lo, hi = float(center), float(lo), float(hi)
            sigma = 0.5 * (hi - lo)
        else:
            raise ValueError(f'unknown err_method={err_method!r}')

        out[m] = dict(center=center, lo=lo, hi=hi, sigma=sigma, n=int(vals.size))

    return out


def latex_sci(x, sig=3):
    if x == 0:
        return "0"

    exponent = int(np.floor(np.log10(abs(x))))
    mantissa = x / 10**exponent

    mantissa_str = f"{mantissa:.{sig}g}"

    if mantissa_str == "1":
        return rf"10^{{{exponent}}}"

    return rf"{mantissa_str} \times 10^{{{exponent}}}"


def band_label(theta_lo, theta_hi):
    """Y-axis label for the band-averaged ratio, carrying the fitted window."""
    return (
        rf'$\left\langle R^{{\rm 3D}}_{{\rm mg}}\right\rangle_'
        rf'{{\theta/R_{{200}} \in [{latex_sci(theta_lo)},\,{latex_sci(theta_hi)}]}}$'
    )


def plot_ratio_spectrum(ax, params, kind='cross_over_gas', models=ALL_MODELS,
                        n_boot=1000, shade_window=True):
    """Left panel: R^3D_mg versus scale, with the averaging window shaded."""
    theta_lo, theta_hi, low_coh, up_coh = params

    for m in models:
        cfg = SIMS[m]
        idx = select_clusters(POWER[m]['coh_3d'], low_coh, up_coh)
        curves = interp_curves(m, theta_grid, idx=idx, kind=kind)

        med, lo, hi = bootstrap_ci(curves, n_boot=n_boot)

        ax.plot(theta_grid, med, linestyle='--', color=cfg['color'], lw=1.6, label=cfg['label'])
        ax.fill_between(theta_grid, lo, hi, color=cfg['color'], alpha=0.15, lw=0)

    if shade_window:
        ax.axvspan(theta_lo, theta_hi, color='0.5', alpha=0.2, zorder=0)

    ax.set_xlabel(r'$\theta / R_{200}$', fontsize=22.5)
    ax.set_ylabel(r'$R^{\rm 3D}_{\rm mg}(\rm \theta = 2\, \pi\, /\, q)$', fontsize=22.5)
    ax.minorticks_on()
    ax.axhline(0.0, color='0.7', lw=0.8, zorder=0)
    ax.set_xscale('log')
    ax.set_xlim(10**-2.2, 1)

    return ax


def plot_band_vs_xsec(ax, summary, params, models=XSEC_MODELS, legend=False):
    """Right panel: <R^3D_mg> over the window versus sigma/m.

    Takes a precomputed `summary` so the same numbers drive the plot and the
    optimizer's loss.
    """
    theta_lo, theta_hi = params[0], params[1]

    for m in models:
        s = summary.get(m)
        if s is None or not np.isfinite(s['center']):
            continue

        cfg = SIMS[m]
        ax.errorbar(
            cfg['xsec'], s['center'],
            yerr=[[s['center'] - s['lo']], [s['hi'] - s['center']]],
            fmt='o', color=cfg['color'], label=cfg['label'], marker=cfg['marker'],
            capsize=5, ms=cfg['ms'], lw=1.5,
        )

    f_size = 22.5
    ax.set_xlabel(r'$\sigma/m$ [cm$^2$ g$^{-1}$]', fontsize=f_size)
    ax.set_ylabel(band_label(theta_lo, theta_hi), fontsize=f_size)
    ax.minorticks_on()

    if legend:
        ax.legend(frameon=False, fontsize=14)

    return ax


def print_summary_table(summary, models=ALL_MODELS):
    """Diagnostic table: value, uncertainty, fractional uncertainty, sample size."""
    print(f"{'model':10s} {'N':>4s} {'<R^3D_mg>':>11s} {'sigma':>9s} {'sigma/|<R>|':>12s}")
    for m in models:
        s = summary[m]
        frac = (s['sigma'] / abs(s['center'])
                if np.isfinite(s['center']) and s['center'] != 0 else np.nan)
        print(f"{m:10s} {s['n']:4d} {s['center']:11.4f} {s['sigma']:9.4f} {frac:12.4f}")

### Optimize the $\theta$ window and the coherence cut

Fits $(\theta_{\rm lo},\, \theta_{\rm hi},\, \ell_{\rm lo},\, \ell_{\rm hi})$ by minimizing a
swappable loss over the three TNG300 runs, which are the same physics at three
resolutions and should therefore give the same
$\left\langle R^{\rm 3D}_{\rm mg}\right\rangle$.

Pass any callable `loss(summary, params) -> float` as `loss_fn`.

In [ ]:
# ------------------------------------------------------------------ #
# Loss functions.
#
# Signature: loss(summary, params) -> float, lower is better.
# The optional `.models` attribute lists the simulations a loss actually needs,
# so the optimizer can skip evaluating the rest.
# ------------------------------------------------------------------ #

def chi2_tng(summary, params=None, precision_weight=0.0):
    """Reduced chi^2 of the three TNG values about their inverse-variance mean.

    TNG300-1/2/3 are the same physics at three resolutions, so their band
    averages should agree: chi2_nu ~ 1 means they are consistent within the
    bootstrap uncertainties, and all three enter symmetrically.

    Caveat: chi^2 falls both when the centers move together and when the
    sigma_i are inflated (a sloppy window with few clusters has large error
    bars). `precision_weight` > 0 adds the mean fractional uncertainty as a
    penalty against that; the diagnostic table below reports sigma/|<R>| either
    way, so the effect stays visible.
    """
    c = np.array([summary[m]['center'] for m in TNG_MODELS], dtype=float)
    s = np.array([summary[m]['sigma'] for m in TNG_MODELS], dtype=float)

    if not np.all(np.isfinite(c)) or not np.all(np.isfinite(s)) or np.any(s <= 0):
        return np.inf

    w = 1.0 / s**2
    c_bar = np.sum(w * c) / np.sum(w)
    chi2_nu = float(np.sum(w * (c - c_bar)**2) / (c.size - 1))

    if precision_weight:
        chi2_nu += precision_weight * float(np.mean(s / np.abs(c_bar)))

    return chi2_nu


def chi2_over_contrast(summary, params=None):
    """chi^2_nu across TNG divided by the CDM -> SIDM 1.0 cm^2/g contrast.

    Rewards windows where the three resolutions agree *and* the SIDM signal is
    large, i.e. resolution consistency per unit of the signal being measured.
    """
    chi2_nu = chi2_tng(summary, params)
    if not np.isfinite(chi2_nu):
        return np.inf

    contrast = abs(summary['CDMb']['center'] - summary['SIDM1b']['center'])
    if not np.isfinite(contrast) or contrast <= 0:
        return np.inf

    return chi2_nu / contrast


chi2_tng.models = TNG_MODELS
chi2_over_contrast.models = TNG_MODELS + ('CDMb', 'SIDM1b')

def fom(summary, params=None, pair=('CDMb', 'SIDM0.1b')):
    c = np.array([summary[m]['center'] for m in TNG_MODELS], float)
    s = np.array([summary[m]['sigma'] for m in TNG_MODELS], float)
    a, b = summary[pair[0]], summary[pair[1]]
    if not np.all(np.isfinite(np.r_[c, s])) or np.any(s <= 0):
        return _PENALTY
    # excess scatter across resolutions, beyond what sampling explains
    var_res = max(0.0, np.var(c, ddof=1) - np.mean(s**2))
    denom = np.sqrt(a['sigma']**2 + b['sigma']**2 + var_res)
    return -abs(a['center'] - b['center']) / denom   # minimize
fom.models = TNG_MODELS + ('CDMb', 'SIDM0.1b')


# ------------------------------------------------------------------ #
# Optimizer.
#
# Internally the fit works with
#     z = [log10(theta_lo), log10(theta_hi / theta_lo), low_coh, up_coh - low_coh]
# i.e. it optimizes the two *widths* rather than the upper edges, so that
# theta_lo < theta_hi and low_coh < up_coh hold by construction under plain box
# bounds.
# ------------------------------------------------------------------ #

def _to_internal(params):
    theta_lo, theta_hi, low_coh, up_coh = params
    return np.array([np.log10(theta_lo), np.log10(theta_hi / theta_lo),
                     low_coh, up_coh - low_coh], dtype=float)


def _from_internal(z):
    theta_lo = 10.0**z[0]
    return (theta_lo, theta_lo * 10.0**z[1], z[2], z[2] + z[3])


INTERNAL_BOUNDS = [
    (np.log10(0.01), np.log10(0.5)),    # theta_lo
    (np.log10(1.15), np.log10(1.5)),   # theta_hi / theta_lo
    (0.0, 0.5),                         # low_coh
    (0.02, 2),                        # up_coh - low_coh
]

_PENALTY = 1e3   # finite, so Nelder-Mead can walk back out of infeasible regions


def _fmt_params(params):
    theta_lo, theta_hi, low_coh, up_coh = params
    return (f'theta = [{theta_lo:.4g}, {theta_hi:.4g}]   '
            f'coh = [{low_coh:.4g}, {up_coh:.4g}]')


def _warn_if_degenerate(loss_value, best_summary, x0_summary, models,
                        tol=1e-3, shrink=0.6):
    """Flag the two ways a chi^2-style loss can be driven down without meaning anything.

    Both are known to bite here, so they are checked every run rather than left
    to the reader.
    """
    msgs = []

    if loss_value < tol:
        msgs.append(
            f'loss = {loss_value:.2e} is essentially zero. Three TNG values carry only '
            'two independent residuals, so four free parameters can solve them exactly; '
            'this optimum is a root, not a preference, and the window is not identifiable. '
            'Treat it as a fit to noise.'
        )

    n_best = np.array([best_summary[m]['n'] for m in models], dtype=float)
    n_x0 = np.array([x0_summary[m]['n'] for m in models], dtype=float)

    if np.all(n_x0 > 0) and np.mean(n_best / n_x0) < shrink:
        msgs.append(
            f'the selected sample shrank from {n_x0.mean():.0f} to {n_best.mean():.0f} '
            'clusters per simulation. chi^2 falls when the error bars grow as well as when '
            'the centers agree, so check sigma/|<R>| above before trusting this window '
            '(chi2_tng(..., precision_weight > 0) penalizes exactly this).'
        )

    for msg in msgs:
        print(f'\nWARNING: {msg}')


def optimize_window(loss_fn=chi2_tng, x0=X0_PARAMS, bounds=INTERNAL_BOUNDS,
                    kind='cross_over_gas', n_band=16, n_boot=300,
                    err_method='bootstrap', min_clusters=10, global_search=False,
                    maxiter=400, seed=1, verbose=True):
    """Fit (theta_lo, theta_hi, low_coh, up_coh) by minimizing `loss_fn`.

    Starts from `x0` -- the hand-picked values by default -- using Nelder-Mead
    with an explicit initial simplex: the default 5% simplex takes steps in
    low_coh far too small to change which clusters are selected, which stalls
    the search on a flat direction.

    Set global_search=True to run differential_evolution first (with x0 seeded
    into the population) and polish the result with Nelder-Mead.

    `n_boot` is deliberately smaller here than for the final figure; the
    bootstrap seed is fixed, so the loss surface is deterministic.
    """
    models = tuple(getattr(loss_fn, 'models', ALL_MODELS))
    history = []

    def objective(z):
        params = _from_internal(np.asarray(z, dtype=float))
        theta_lo, theta_hi, low_coh, up_coh = params

        if not np.all(np.isfinite(params)) or theta_hi > 1.5 or up_coh <= low_coh:
            return _PENALTY

        summary = band_summary(params, models=models, kind=kind, n_band=n_band,
                               n_boot=n_boot, err_method=err_method)

        n_min = min(summary[m]['n'] for m in models)
        if n_min < min_clusters:
            # Sloped penalty so the search is pushed back towards feasibility.
            return _PENALTY * (1.0 + (min_clusters - n_min) / min_clusters)

        value = loss_fn(summary, params)
        if not np.isfinite(value):
            return _PENALTY

        history.append((params, float(value)))
        return float(value)

    z0 = _to_internal(x0)
    loss_x0 = objective(z0)

    z_start = z0
    if global_search:
        de = differential_evolution(objective, bounds, x0=z0, seed=seed,
                                    maxiter=60, tol=1e-6, polish=False)
        z_start = de.x

    lo_b = np.array([b[0] for b in bounds])
    hi_b = np.array([b[1] for b in bounds])

    steps = np.array([0.15, 0.15, 0.05, 0.10])
    simplex = np.vstack([z_start] + [z_start + np.eye(4)[i] * steps[i] for i in range(4)])
    simplex = np.clip(simplex, lo_b, hi_b)

    res = minimize(objective, z_start, method='Nelder-Mead', bounds=bounds,
                   options=dict(initial_simplex=simplex, maxiter=maxiter,
                                xatol=1e-4, fatol=1e-6))

    best_params = _from_internal(res.x)
    best_summary = band_summary(best_params, models=ALL_MODELS, kind=kind,
                                n_band=n_band, n_boot=1000)

    if verbose:
        x0_summary = band_summary(tuple(x0), models=ALL_MODELS, kind=kind,
                                  n_band=n_band, n_boot=1000)

        print(f'loss   at x0 = {loss_x0:.6g}    {_fmt_params(x0)}')
        print(f'loss at best = {res.fun:.6g}    {_fmt_params(best_params)}')
        print(f'nfev = {res.nfev}   success = {res.success}   {res.message}')

        print('\n--- initial guess ---')
        print_summary_table(x0_summary)
        print('\n--- optimum ---')
        print_summary_table(best_summary)

        _warn_if_degenerate(res.fun, best_summary, x0_summary, models)

    return dict(params=best_params, loss=float(res.fun), loss_x0=float(loss_x0),
                summary=best_summary, result=res, history=history)


# OPT = optimize_window(loss_fn=chi2_tng, x0=X0_PARAMS, n_boot=300, min_clusters=10)
#OPT = optimize_window(loss_fn=chi2_over_contrast, x0=X0_PARAMS, global_search=True)
OPT = optimize_window(loss_fn=fom, x0=X0_PARAMS, n_boot=300, min_clusters=10)

BEST_PARAMS = OPT['params']

In [ ]:
# Parameters for both panels. Swap in X0_PARAMS to reproduce the hand-picked
# version of this figure.
PARAMS = BEST_PARAMS
#PARAMS = X0_PARAMS
#PARAMS = (0.05807, 0.07549, 0.04262, 0.4125)

summary = band_summary(PARAMS, models=ALL_MODELS, n_boot=1000)

fig, (ax_spec, ax_xsec) = plt.subplots(1, 2, figsize=(12, 5))

# Left: cross/gas power ratio vs scale, with the averaging window shaded.
plot_ratio_spectrum(ax_spec, PARAMS, n_boot=1000)

# Right: <R^3D> over that window vs sigma/m (vdSIDM has no single sigma/m).
plot_band_vs_xsec(ax_xsec, summary, PARAMS)

# Shared legend from the spectrum panel, which does include vdSIDM.
handles, labels = ax_spec.get_legend_handles_labels()
ax_xsec.legend(
    handles, labels,
    loc='center left',
    bbox_to_anchor=(1.03, 0.43),
    fontsize=15,
    labelspacing=1.15,
)

fig.tight_layout()
fig.savefig(f'{figure_path}/dm_gas_power_ratio.png', bbox_inches='tight', dpi=350)
plt.show()

# Cluster sample: 3D coherence length and $M_{200}$ distributions

Diagnostic for the sample that enters Figure 1 -- not letter-ready. The left panel
shows every cluster with the fitted coherence-length window shaded; the right panel
shows the masses of the clusters that window actually selects.

### Import $M_{200}$

In [ ]:
M200 = {}

for _model in TNG_MODELS:
    _m200_all = iapi.getHaloField(
        field='Group_M_Crit200', simulation=_model, snapshot=99,
        fileName=TNG_data_path + 'TNG_data/' + _model + '_Group_M_Crit200',
        rewriteFile=0,
    )
    M200[_model] = np.log10(_m200_all[:100] * 10**10 / h)

# We may need to add a factor of data['h'] to the BAHAMAS masses; come back here
# if something looks weird -- np.load(bahamas_path + f"/CDMb/GrNm_{i}.npz")['h']
for _model in ('CDMb', 'SIDM0.1b', 'SIDM0.3b', 'SIDM1b', 'vdSIDMb'):
    M200[_model] = np.log10(np.array([
        np.load(f'{bahamas_path}/{_model}/GrNm_{i}.npz')['M200'] for i in cluster_ids
    ]))

In [ ]:
low_coh, up_coh = BEST_PARAMS[2], BEST_PARAMS[3]

coherence_bins = np.linspace(0.0, 1.0, 50)
mass_bins = np.linspace(14.0, 15.25, 50)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True, constrained_layout=False)
ax_coherence, ax_mass = axes

for model in ALL_MODELS:
    cfg = SIMS[model]
    coh_3d = POWER[model]['coh_3d']
    idx = select_clusters(coh_3d, low_coh, up_coh)

    # Left: the full sample, so the shaded window shows what the cut removes.
    # The dashed line is the median of the *selected* clusters.
    ax_coherence.hist(
        coh_3d[np.isfinite(coh_3d)],
        bins=coherence_bins,
        linewidth=2.0,
        color=cfg['color'],
        alpha=0.5,
        label=cfg['label'],
    )
    ax_coherence.axvline(
        np.nanmedian(coh_3d[idx]),
        linestyle='--',
        linewidth=2.0,
        color=cfg['color'],
    )

    # Right: masses of the clusters that survive the coherence cut, i.e. the
    # sample Figure 1 is built from.
    mass = M200[model][idx]
    mass = mass[np.isfinite(mass)]

    ax_mass.hist(
        mass,
        bins=mass_bins,
        linewidth=2.0,
        alpha=0.5,
        color=cfg['color'],
        label=cfg['label'],
    )
    ax_mass.axvline(
        np.nanmedian(mass),
        linestyle='--',
        linewidth=2.0,
        color=cfg['color'],
    )

# Fitted selection window.
ax_coherence.axvspan(low_coh, up_coh, color='0.5', alpha=0.2, zorder=0)

ax_coherence.set_xlabel("3D coherence length", fontsize=14, fontweight="semibold")
ax_coherence.set_ylabel("Number of clusters", fontsize=14, fontweight="semibold")
ax_coherence.set_xlim(coherence_bins[0], coherence_bins[-1])

ax_mass.set_xlabel(r"$\log_{10}\!\left(M_{200}/M_{\odot}\right)$",
                   fontsize=14, fontweight="semibold")
ax_mass.set_xlim(mass_bins[0], mass_bins[-1])

for ax in axes:
    ax.tick_params(axis="both", which="both", direction="in",
                   top=True, right=True, labelsize=12)
    ax.grid(alpha=0.2, linestyle="--")

handles, labels = ax_coherence.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.1),
           ncol=4, frameon=False, fontsize=20)

fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.72, wspace=0.08)

plt.show()

# Extra Figure: $C^{\rm 3D}$ vs scale (log-log and linear-log)

Uses the power/coherence stacks and the `interp_curves` / `bootstrap_ci` helpers
imported and defined for Figure 1, with `kind='coherence'`. All clusters are
included -- no coherence cut is applied here.

In [ ]:
fig, (ax_coh_log, ax_coh_lin) = plt.subplots(1, 2, figsize=(12, 5))

for model in ALL_MODELS:
    cfg = SIMS[model]
    cg = interp_curves(model, theta_grid, kind='coherence')

    med, lo, hi = bootstrap_ci(cg, n_boot=10**3, ci=(16, 84), random_state=123)

    for ax in (ax_coh_log, ax_coh_lin):
        ax.plot(theta_grid, med, linestyle='--', color=cfg['color'], lw=1.6, label=cfg['label'])
        ax.fill_between(theta_grid, lo, hi, color=cfg['color'], alpha=0.15, lw=0)

for ax in (ax_coh_log, ax_coh_lin):
    ax.set_xlabel(r'$\theta / R_{200}$', fontsize=18)
    ax.set_ylabel(r'$C^{\rm 3D}(\rm \theta = 2\, \pi\, /\, q)$', fontsize=18)
    ax.minorticks_on()
    ax.axhspan(0.9, 20, color='0.5', alpha=0.2, zorder=0)

ax_coh_log.set_ylim(10**-4, 1.05)
ax_coh_log.set_xlim(10**-2.2, 1.5)
ax_coh_log.set_yscale('symlog', linthresh=1e-2)
ax_coh_log.set_xscale('log')

ax_coh_lin.set_ylim(0.0, 1.05)
ax_coh_lin.set_xlim(10**-2.2, 1.5)
ax_coh_lin.set_xscale('log')

handles, labels = ax_coh_lin.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax_coh_lin.legend(
    by_label.values(),
    by_label.keys(),
    loc='center left',
    bbox_to_anchor=(1.03, 0.5),
    fontsize=12,
    labelspacing=1.5,
)

fig.tight_layout()
fig.savefig(f'{figure_path}/coherence_vs_scale.png', bbox_inches='tight', dpi=350)
plt.show()

# Appendix Figure: $|\,l_{\rm CR}^{3\mathrm{D}}-l_{\rm CR}^{2\mathrm{D}}\,|$ vs $M_{200}$ and triaxiality

For each simulation we have $100\!\times\!1000$ projection-level absolute differences.
Triaxiality $T$ is computed from the 3D positions of cluster members via
`TNG_DA.shape_index_3d` and broadcast across projections (it is invariant to viewing
direction). Only TNG300-1 is plotted.

**Missing input data.** The triaxiality panel needs three TNG300-1 subhalo caches that
are not currently on disk:
`TNG300-1_SubhaloPos.hdf5`, `TNG300-1_SubhaloGrNr.hdf5` and
`TNG300-1_SubhaloStellarPhotometrics.hdf5` under `/home/habjan.e/TNG/Data/TNG_data/`.
The import cell below reads them from that path as before and will fail until they are
restored; regenerate them with
`iapi.getSubhaloField(field=..., simulation='TNG300-1', snapshot=99, fileName=...)`.

### Import 2D coherence data and TNG300-1 subhalo catalogs

In [ ]:
# 2D (per-projection) coherence lengths, shape (Ncluster, Nproj).
_COH_2D_DIRS = {
    'CDMb':     'CDMb',
    'SIDM0.1b': 'SIDM0.1b',
    'SIDM0.3b': 'SIDM0.3b',
    'SIDM1b':   'SIDM1b',
    'vdSIDMb':  'vdSIDMb',
    'TNG300-1': 'TNG',
}

COH_2D = {}
COH_2D_ERR = {}

for _model, _subdir in _COH_2D_DIRS.items():
    _ids = SIMS[_model]['ids']
    COH_2D[_model] = np.array([
        np.load(f'{base_coh_path}/{_subdir}/coherence_length_{i}.npy') for i in _ids
    ])
    COH_2D_ERR[_model] = np.array([
        np.load(f'{base_coh_path}/{_subdir}/coherence_length_err_{i}.npy') for i in _ids
    ])


def _h5_array(path, dataset):
    with h5py.File(path, 'r') as f:
        return f[dataset][...]


# TNG300-1 subhalo catalogs, read straight from the local hdf5 caches so we never
# depend on the TNG API (which can return 404 / time out).
_TNG_CACHE = TNG_data_path + 'TNG_data/' + 'TNG300-1'

_h_tng        = 0.6774
_GroupPos     = _h5_array(_TNG_CACHE + '_GroupPos.hdf5',                   'Group/GroupPos')
_SubhaloPos   = _h5_array(_TNG_CACHE + '_SubhaloPos.hdf5',                 'Subhalo/SubhaloPos')
_SubhaloPhoto = _h5_array(_TNG_CACHE + '_SubhaloStellarPhotometrics.hdf5', 'Subhalo/SubhaloStellarPhotometrics')
_SubhaloGrNr  = _h5_array(_TNG_CACHE + '_SubhaloGrNr.hdf5',                'Subhalo/SubhaloGrNr')

_TNG_BOX_CKPC = 205000.0   # comoving kpc, full TNG300 box

### Functions

In [ ]:
def _tng_member_positions(cluster_ind, mag_cut=-18):
    """Cluster-centered positions of the bright TNG300-1 members, in kpc."""
    cm = _GroupPos[cluster_ind] / _h_tng
    sub_pos = _SubhaloPos / _h_tng

    in_grp = np.where(_SubhaloGrNr == cluster_ind)[0]
    pos = sub_pos[in_grp] - cm

    halfbox = (_TNG_BOX_CKPC / _h_tng) / 2.0
    pos = (pos + halfbox) % (2.0 * halfbox) - halfbox

    bright = _SubhaloPhoto[in_grp, 4] < mag_cut
    return pos[bright]


def _triax_from_positions(pos_3d):
    """Triaxiality T from member positions; depends only on the 3D shape, so it
    is identical for every projection of a given cluster."""
    if pos_3d.shape[0] < 10:
        return np.nan

    _, T = TNG_DA.shape_index_3d(pos_3d)
    return float(T)


def _abs_diff_stack(coh_2d, coh_3d):
    """|3D - 2D| with shape (Ncluster, Nproj). 3D is broadcast across projections."""
    return np.abs(coh_3d[:, None] - coh_2d)


def _finite_percentile_bins(arr, n, lo=1, hi=99):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.linspace(np.nanpercentile(arr, lo), np.nanpercentile(arr, hi), n + 1)


def _binned_arrays(x, y, xbins, min_count=5):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    out = []
    centers = 0.5 * (xbins[:-1] + xbins[1:])

    for i, (lo, hi) in enumerate(zip(xbins[:-1], xbins[1:])):
        if i == len(xbins) - 2:
            m = np.isfinite(x) & np.isfinite(y) & (x >= lo) & (x <= hi)
        else:
            m = np.isfinite(x) & np.isfinite(y) & (x >= lo) & (x < hi)

        yy = y[m]
        out.append(yy if yy.size >= min_count else np.array([]))

    return centers, out


def _plot_plain_binned_violin(ax, x, y, xbins, *, ylabel, ylim=None,
                              xlabel=r'$\log_{10}(M_{200}/M_\odot)$'):
    centers, data = _binned_arrays(x, y, xbins)

    widths = 0.75 * np.diff(xbins)
    widths = np.full_like(centers, np.nanmedian(widths), dtype=float)

    valid = [d for d in data if len(d) > 0]
    valid_pos = [c for c, d in zip(centers, data) if len(d) > 0]

    # Scale each violin's width by its total count relative to the most-
    # populated bin, so the visual "amount" reflects how many points it holds
    # (matplotlib otherwise normalizes every violin to the same width).
    valid_counts = np.array([len(d) for d in valid], dtype=float)
    base_width = np.nanmedian(widths)
    valid_widths = list(base_width * valid_counts / valid_counts.max())

    parts = ax.violinplot(
        valid,
        positions=valid_pos,
        widths=valid_widths,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for body in parts['bodies']:
        body.set_facecolor('0.75')
        body.set_edgecolor('black')
        body.set_alpha(0.8)
        body.set_linewidth(0.8)

    # Connect the per-bin medians with a dashed red line and shade +/- 1 std.
    med_color = '#c1121f'
    medians = np.array([np.nanmedian(d) for d in valid], dtype=float)
    stds = np.array([np.nanstd(d) for d in valid], dtype=float)
    pos = np.asarray(valid_pos, dtype=float)

    pos = np.concatenate(([xbins[0]], pos, [xbins[-1]]))
    medians = np.concatenate((medians[:1], medians, medians[-1:]))
    stds = np.concatenate((stds[:1], stds, stds[-1:]))

    ax.fill_between(pos, medians - stds, medians + stds,
                    color=med_color, alpha=0.2, linewidth=0, zorder=2)
    ax.plot(pos, medians, linestyle='--', color=med_color, linewidth=3.5, zorder=3)

    ax.set_xticks(centers)
    ax.set_xticklabels([f'{c:.2f}' for c in centers])

    for b in xbins:
        ax.axvline(b, color='0.9', lw=0.8, zorder=0)

    f_size = 20
    ax.set_xlabel(xlabel, fontsize=f_size)
    ax.set_ylabel(ylabel, fontsize=f_size)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.minorticks_on()


tng_T = np.array([
    _triax_from_positions(_tng_member_positions(int(i))) for i in cluster_ids_tng
])

In [ ]:
# Per-projection |3D - 2D| for TNG300-1: shape (Ncluster, Nproj).
tng_diff = _abs_diff_stack(COH_2D['TNG300-1'], POWER['TNG300-1']['coh_3d'])

# Broadcast per-cluster M_200 and T across all projections, then flatten so each
# point is one projection measurement.
tng_m_flat = np.repeat(M200['TNG300-1'], tng_diff.shape[1])
tng_T_flat = np.repeat(tng_T, tng_diff.shape[1])
tng_d_flat = tng_diff.ravel()

bins_violin_m = np.linspace(14.25, 15.25, 6)
bins_violin_t = _finite_percentile_bins(tng_T_flat, 5)

YLABEL_DIFF = r'$|\ell_{\rm 3D} - \ell_{\rm 2D}|/R_{200}$'

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, constrained_layout=True)
ax_m, ax_t = axes

_plot_plain_binned_violin(
    ax_m, tng_m_flat, tng_d_flat, bins_violin_m,
    ylabel=YLABEL_DIFF,
    xlabel=r'$\log_{10}(M_{200}/M_\odot)$',
)

_plot_plain_binned_violin(
    ax_t, tng_T_flat, tng_d_flat, bins_violin_t,
    ylabel='',
    xlabel=r'Triaxiality $T$',
)

fig.savefig(f'{figure_path}/coh_len_violin.png', bbox_inches='tight', dpi=500)
plt.show()